# Region analysis on synthetic label volumes

Build two labeled 3D volumes, extract region properties with `RegionAnalyzer`, then filter regions with `RegionFilter`.

In [1]:
import numpy as np
import pandas as pd
import stackview
from scipy.ndimage import gaussian_filter

from vistiq.utils import ArrayIteratorConfig
from vistiq.matrix.types import FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, MinFilterConfig, RangeFilterConfig
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig

2026-07-12 09:32:38,246 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


## Synthetic volumes

Two `(10, 200, 200)` uint64 volumes (Z×Y×X):

- **`img`** — five non-overlapping objects
- **`labels`** — the corresponding regions with labels `6` and `8`

In [2]:
rng = np.random.default_rng(0)
regions = {
    1: (slice(2, 7),   slice(20, 38),   slice(12, 30)),
    2: (slice(3, 9),   slice(42, 58),   slice(38, 62)),
    3: (slice(1, 6),   slice(72, 92),   slice(68, 88)),
    4: (slice(4, 6),   slice(112, 132), slice(125, 163)),
    5: (slice(7, 9),   slice(145, 180), slice(12, 58)),
}
# Per-object smooth baseline intensity (signal level)
baseline = {1: 120, 2: 160, 3: 90, 4: 200, 5: 140}
shape = (10, 200, 200)

# 1) Smooth per-object baseline (no noise yet)
signal = np.zeros(shape, dtype=np.float32)
for label, sl in regions.items():
    signal[sl] = baseline[label]
    
# 2) Optional blur of the baseline -> soft object edges (partial-volume look).
#    sigma is per-axis (Z, Y, X); Z blurred less since z-resolution is coarser.
signal = gaussian_filter(signal, sigma=(0.5, 1.5, 1.5))

# 3) Per-voxel Gaussian noise:
#    - low-level background/read noise everywhere
#    - extra signal-dependent (shot-like) noise inside objects
background_noise_std = 2.0
signal_noise_frac = 0.10  # noise std as fraction of local intensity
noise = rng.normal(0.0, background_noise_std, size=shape).astype(np.float32)
noise += rng.normal(0.0, 1.0, size=shape).astype(np.float32) * (signal_noise_frac * signal)
img = signal + noise

# 4) Add a small constant background offset (detector floor), clip, cast
img += 10.0
img = np.clip(img, 0, 255).astype(np.uint8)

In [3]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 42:58, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:58] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [4]:
stackview.slice(img)

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [30]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    #map_slices=True,
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
        "intensity_min",
        "intensity_max",
        "intensity_mean",
        "euler_number",
        "slice_annotations",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

regions = RegionAnalyzer(config).run(labels,img, metadata=metadata)
print (f"Allowed properties: {RegionAnalyzer.allowed_properties()}")

2026-07-12 10:16:00,913 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-12 10:16:00,964 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-12 10:16:00,995 - INFO - Running RegionAnalyzer with config: classname='vistiq.core.Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None index_on='label' properties=['label', 'stack_id', 'slice_id', 'object_id', 'centroid', 'bbox', 'aspect_ratio', 'cross_sectional_area', 'i

Allowed properties: ['area', 'area_bbox', 'area_convex', 'area_filled', 'aspect_ratio', 'axis_major_length', 'axis_minor_length', 'bbox', 'centroid', 'centroid_local', 'centroid_weighted', 'centroid_weighted_local', 'channel', 'circularity', 'coords', 'coords_scaled', 'cross_sectional_area', 'eccentricity', 'equivalent_diameter_area', 'euler_number', 'extent', 'feret_diameter_max', 'image', 'image_convex', 'image_filled', 'image_intensity', 'inertia_tensor', 'inertia_tensor_eigvals', 'intensity_max', 'intensity_mean', 'intensity_min', 'intensity_std', 'label', 'moments', 'moments_central', 'moments_hu', 'moments_normalized', 'moments_weighted', 'moments_weighted_central', 'moments_weighted_hu', 'moments_weighted_normalized', 'num_pixels', 'object_id', 'object_name', 'orientation', 'perimeter', 'perimeter_crofton', 'slice', 'slice_annotations', 'slice_id', 'solidity', 'sphericity', 'stack_id', 'volume']


In [29]:
regions

,centroid-y,centroid-x,bbox-start-y,bbox-start-x,bbox-end-y,bbox-end-x,intensity_min,intensity_max,intensity_mean,euler_number,area,aspect_ratio,cross_sectional_area,z,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,
3,81.5,77.5,72,68,92,88,39.0,115.0,81.035000,1,400.0,1.000000,400.0,1,ebd5f5d2dc9e4cf1bcfd1667ebf52cf2,83381c206d8c4baf9a44345658fea6e7,c981c937955d47c0b6da13f4c5ab29bf
1,28.5,20.5,20,12,38,30,45.0,144.0,103.194444,1,324.0,1.000000,324.0,2,9e1248a143994a4e999a239083d89415,83381c206d8c4baf9a44345658fea6e7,0f4799c3072f479a9902409f14b13e21
3,81.5,77.5,72,68,92,88,45.0,125.0,89.175000,1,400.0,1.000000,400.0,2,c8a54786260b4167ad90830eb2224181,83381c206d8c4baf9a44345658fea6e7,0f4799c3072f479a9902409f14b13e21
1,28.5,20.5,20,12,38,30,55.0,161.0,114.475309,1,324.0,1.000000,324.0,3,d823d49ba46a43aba1f382971efc06e9,83381c206d8c4baf9a44345658fea6e7,3755405ba60b4f17bde26bac31e7dc76
2,49.5,49.5,42,38,58,62,61.0,210.0,135.809896,1,384.0,0.665942,384.0,3,fc59e25c50ce42818f0ebb3b83678cc9,83381c206d8c4baf9a44345658fea6e7,3755405ba60b4f17bde26bac31e7dc76
3,81.5,77.5,72,68,92,88,44.0,124.0,90.052500,1,400.0,1.000000,400.0,3,96a9f51faf8646a0a14095334593d34b,83381c206d8c4baf9a44345658fea6e7,3755405ba60b4f17bde26bac31e7dc76
1,28.5,20.5,20,12,38,30,46.0,177.0,114.669753,1,324.0,1.000000,324.0,4,dda6edd54b31405daf129f1f3fd36c99,83381c206d8c4baf9a44345658fea6e7,eda488d2d9a0493eb80c474a7f5872e7
2,49.5,49.5,42,38,58,62,73.0,216.0,151.388021,1,384.0,0.665942,384.0,4,a6d5723946f04cc192cfba188d8b879e,83381c206d8c4baf9a44345658fea6e7,eda488d2d9a0493eb80c474a7f5872e7
3,81.5,77.5,72,68,92,88,43.0,129.0,89.587500,1,400.0,1.000000,400.0,4,9b1eff72bf0a439fbed3b77ce80928cc,83381c206d8c4baf9a44345658fea6e7,eda488d2d9a0493eb80c474a7f5872e7


# Filter Regions

In [23]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(1000.0,np.inf)
        ),
        MinFilterConfig(
            attribute="intensity_min",
            minimum=50,
        ),
    ]
)
accepted, _ = RegionFilter(rfcfg).run(regions)

2026-07-12 10:09:46,951 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-12 10:09:47,032 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-12 10:09:47,039 - INFO - Running RegionFilter with config: classname='vistiq.core.Configurable' package='vistiq.core' version=None command_group=None filters=[RangeFilterConfig(classname='vistiq.core.Configurable', package='vistiq.core', version=None, command_group=None, attribute='volume', axis=None, strict=True, preferred_input_type='np.ndarray', range=(1000.0, inf)), MinFilterConfig(classname='vistiq.core.Configurable', package='vistiq.core', version=None, command_group=None, attribute='intensity_min', axis=None, strict=True, preferred_input_type='np.ndar

In [24]:
accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,intensity_min,...,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,,
2,11.0,49.5,49.5,3,42,38,9,58,62,61.0,...,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,927eecd6d2c64e04a0c704d26e45b1a8,5d547834611d4ccb8bfcf8db91514b70,5dea628fbd8046598f13075403730f6a
4,9.0,121.5,143.5,4,112,125,6,132,163,70.0,...,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,9b3663e283694fcba97c31b2363251c9,5d547834611d4ccb8bfcf8db91514b70,5dea628fbd8046598f13075403730f6a
5,15.0,162.0,34.5,7,145,12,9,180,58,50.0,...,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,e8806f9e859040039bc163f15c7140d4,5d547834611d4ccb8bfcf8db91514b70,5dea628fbd8046598f13075403730f6a
